In [0]:
from pyspark.sql import functions as F

# El comodín * toma todos los meses que haya en el volume
ruta = "/Volumes/oece/bronze/raw/*_seace_v3.json"

df_raw = spark.read.option("multiLine", True).json(ruta)

df_bronze = (
    df_raw
    .select(
        F.explode("records").alias("r"),
        F.col("_metadata.file_path").alias("_archivo_origen"),
    )
    .select(
        F.col("r.ocid").alias("ocid"),
        F.col("r.compiledRelease").alias("compiled_release"),
        F.regexp_extract("_archivo_origen", r"(\d{4}-\d{2})_seace", 1).alias("periodo"),
        "_archivo_origen",
        F.current_timestamp().alias("_fecha_ingesta"),
    )
)

(df_bronze.write
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable("oece.bronze.seace_records"))

display(
    spark.table("oece.bronze.seace_records")
    .groupBy("periodo").count().orderBy("periodo")
)

periodo,count
2026-08,6713
2026-09,6752
2026-10,6620
2026-11,1546
